## Core Concepts of injector

Dependency injection in Python is built on a few foundational concepts. These concepts help organize code, decouple components, and make applications more maintainable. Here’s a breakdown of the core concepts in the injector library.

--- 

### Modules

Modules in injector define how dependencies are created and managed. They act as containers for bindings, which connect a dependency (e.g., a class or interface) to its implementation or instance. Modules centralize dependency logic and keep the application loosely coupled.

#### Learn How to Create and Bind Objects in Modules

You can define a module by subclassing the Module class and overriding its configure method to define bindings.

##### Example: Binding Configuration Settings

In [ ]:
from injector import Module, Injector, singleton

# Define a class for configuration settings
class AppConfig:
    def __init__(self, app_name: str, version: str):
        self.app_name = app_name
        self.version = version

# Define a module for bindings
class AppModule(Module):
    def configure(self, binder):
        # Bind AppConfig to a specific instance
        binder.bind(AppConfig, to=AppConfig("MyApp", "1.0"), scope=singleton)

# Create an injector with the module
injector = Injector([AppModule()])

# Retrieve the bound AppConfig instance
config = injector.get(AppConfig)

print(config.app_name)  # Output: MyApp
print(config.version)   # Output: 1.0


#### Binding Classes and Interfaces

You can bind an abstract base class or interface to a concrete implementation, ensuring loose coupling between components.

##### Example: Binding an Interface to an Implementation

In [ ]:
from abc import ABC, abstractmethod
from injector import Module, Injector

# Define an abstract base class
class Database(ABC):
    @abstractmethod
    def fetch_data(self):
        pass

# Concrete implementation of the abstract class
class MySQLDatabase(Database):
    def fetch_data(self):
        return "Fetched data from MySQL"

# Module to bind the interface to the implementation
class DatabaseModule(Module):
    def configure(self, binder):
        binder.bind(Database, to=MySQLDatabase)

# Use the injector
injector = Injector([DatabaseModule()])
db = injector.get(Database)
print(db.fetch_data())  # Output: Fetched data from MySQL



---

### @inject decorator
The @inject decorator marks a class constructor or method as injectable. It allows the injector library to automatically resolve dependencies and inject them when an instance is created or a method is called.

#### Understand Constructor Injection

Dependencies can be injected into a class constructor, making them available as attributes.

##### Example: Constructor Injection

In [ ]:
from injector import inject, Module, Injector

class Database:
    def fetch_data(self):
        return "Fetched data from the database"

class ReportGenerator:
    @inject
    def __init__(self, database: Database):
        self.database = database

    def generate(self):
        print(f"Report generated using: {self.database.fetch_data()}")

# Define a module to bind the Database dependency
class AppModule(Module):
    def configure(self, binder):
        binder.bind(Database)

# Use the injector
injector = Injector([AppModule()])
report_generator = injector.get(ReportGenerator)
report_generator.generate()


#### Understand Method Injection

Dependencies can also be injected directly into methods for specific use cases.

##### Example: Method Injection

In [ ]:
from injector import Module, Injector, inject

class Service:
    def execute(self):
        print("Service executed!")

class Task:
    @inject
    def run(self, service: Service):
        service.execute()

# Bind the Service class
class TaskModule(Module):
    def configure(self, binder):
        binder.bind(Service)

injector = Injector([TaskModule()])
task = injector.get(Task)

# Use call_with_injection to run the method with dependencies resolved
injector.call_with_injection(task.run)


**Explanation**

The @inject decorator only works with constructors or Injector.call_with_injection for methods. If you call the method directly (task.run()), Python expects you to manually pass all required arguments, including the service parameter.

---

### Key Features

#### Explicit Bindings

Explicit bindings define specific implementations for dependencies. Use binder.bind() in a module to explicitly specify how a dependency should be resolved.

##### Example: Explicit Binding

In [ ]:
from injector import Module, Injector

class MyService:
    def perform_task(self):
        print("Task performed!")

class AppModule(Module):
    def configure(self, binder):
        binder.bind(MyService, to=MyService())

injector = Injector([AppModule()])
service = injector.get(MyService)
service.perform_task()


#### Implicit Bindings

If no explicit binding is provided, the injector library can automatically create instances of concrete classes, provided they have an injectable constructor and resolvable dependencies.

##### Example: Implicit Binding

In [ ]:
from injector import Injector

class Logger:
    def log(self, message: str):
        print(f"Log: {message}")

# Injector automatically resolves Logger without explicit binding
injector = Injector()
logger = injector.get(Logger)
logger.log("Implicit binding example!")



---

### Scopes

Scopes control the lifecycle of dependencies. injector provides various scopes to manage object lifetimes:

- Singleton Scope: A single instance is created and shared throughout the application.

- No Scope: A new instance is created each time the dependency is resolved.


#### Explore the Singleton Scope

In a singleton scope, the same instance of a dependency is reused for all requests.

##### Example: Singleton Scope


In [ ]:
from injector import Injector, Module, singleton

class Service:
    def __init__(self):
        print("Service initialized!")

class ServiceModule(Module):
    def configure(self, binder):
        binder.bind(Service, scope=singleton)

injector = Injector([ServiceModule()])
service1 = injector.get(Service)
service2 = injector.get(Service)

print(service1 is service2)  # Output: True


#### Explore the No Scope (Default Behavior)

By default, each request for a dependency creates a new instance.

##### Example: No Scope

In [ ]:
from injector import Injector

class AnotherService:
    def __init__(self):
        print("AnotherService initialized!")

injector = Injector()
service1 = injector.get(AnotherService)
service2 = injector.get(AnotherService)

print(service1 is service2)  # Output: False



---

### Summary

- Modules: Define and centralize dependency bindings, making applications modular and maintainable.

- @inject Decorator: Automatically inject dependencies into constructors or methods.

- Key Features:

    - Explicit Bindings: Provide control over dependency resolution.

    - Implicit Bindings: Simplify resolving dependencies without explicit configuration.

- Scopes: Manage object lifetimes, with support for singleton and other scope types.